# Proyecto BI - Uber NYC (2009-2015)
## Fase 3: Preparacion y Modelado de Datos
Curso: Analisis de Datos - ITM 2026-1

Este notebook limpia, transforma y prepara el dataset para dashboard en Power BI o Tableau.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

def print_section(title: str):
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)

### 1) Carga del dataset

In [ ]:
print_section("CARGA DEL DATASET")

df = pd.read_csv("uber.csv")

if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

required_columns = [
    "fare_amount", "pickup_datetime",
    "pickup_longitude", "pickup_latitude",
    "dropoff_longitude", "dropoff_latitude",
    "passenger_count"
]
missing = [c for c in required_columns if c not in df.columns]
if missing:
    raise ValueError(f"Faltan columnas requeridas: {missing}")

print(f"Filas originales: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")
print(f"\nTipos de datos originales:\n{df.dtypes}")
print(f"\nValores nulos por columna:\n{df.isnull().sum()}")
print(f"\nPrimeras filas:\n{df.head(3)}")

### 2) Limpieza de datos

In [ ]:
print_section("LIMPIEZA DE DATOS")

filas_antes = len(df)

df.drop_duplicates(inplace=True)
print(f"Duplicados eliminados: {filas_antes - len(df):,}")

numeric_critical_columns = [
    "fare_amount", "pickup_longitude", "pickup_latitude",
    "dropoff_longitude", "dropoff_latitude", "passenger_count"
]
for col in numeric_critical_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

critical_columns = ["pickup_datetime"] + numeric_critical_columns
df.dropna(subset=critical_columns, inplace=True)
print(f"Filas tras eliminar nulos: {len(df):,}")

df["pickup_datetime"] = pd.to_datetime(df["pickup_datetime"], utc=True, errors="coerce")
df.dropna(subset=["pickup_datetime"], inplace=True)

mask_fare = (df["fare_amount"] > 0) & (df["fare_amount"] <= 200)
eliminados_fare = (~mask_fare).sum()
df = df[mask_fare]
print(f"Filas eliminadas por tarifa invalida (<=0 o >200): {eliminados_fare:,}")

def coordenadas_validas_nyc(lat, lon):
    lat_min, lat_max = 40.4774, 40.9176
    lon_min, lon_max = -74.2591, -73.7004
    return (lat >= lat_min) & (lat <= lat_max) & (lon >= lon_min) & (lon <= lon_max)

mask_pickup = coordenadas_validas_nyc(df["pickup_latitude"], df["pickup_longitude"])
mask_dropoff = coordenadas_validas_nyc(df["dropoff_latitude"], df["dropoff_longitude"])
eliminados_coords = (~(mask_pickup & mask_dropoff)).sum()
df = df[mask_pickup & mask_dropoff].copy()
print(f"Filas eliminadas por coordenadas invalidas: {eliminados_coords:,}")

mask_pax = (df["passenger_count"] >= 1) & (df["passenger_count"] <= 8)
eliminados_pax = (~mask_pax).sum()
df = df[mask_pax].copy()
print(f"Filas eliminadas por passenger_count invalido: {eliminados_pax:,}")

print(f"\nFilas finales tras limpieza: {len(df):,}")
print(f"Porcentaje de datos conservados: {len(df) / filas_antes * 100:.1f}%")

### 3) Transformacion de datos

In [ ]:
print_section("TRANSFORMACION DE DATOS")

df["year"] = df["pickup_datetime"].dt.year
df["month"] = df["pickup_datetime"].dt.month
df["day_of_week"] = df["pickup_datetime"].dt.dayofweek
df["hour"] = df["pickup_datetime"].dt.hour
df["month_name"] = df["pickup_datetime"].dt.strftime("%B")
df["day_name"] = df["pickup_datetime"].dt.strftime("%A")

def franja_horaria(h):
    if 0 <= h < 6:
        return "Madrugada (0-5h)"
    if 6 <= h < 12:
        return "Manana (6-11h)"
    if 12 <= h < 18:
        return "Tarde (12-17h)"
    return "Noche (18-23h)"

df["time_of_day"] = df["hour"].apply(franja_horaria)

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

df["distance_km"] = haversine(
    df["pickup_latitude"].values,
    df["pickup_longitude"].values,
    df["dropoff_latitude"].values,
    df["dropoff_longitude"].values,
)

mask_dist = df["distance_km"] >= 0.1
eliminados_dist = (~mask_dist).sum()
df = df[mask_dist].copy()
print(f"Filas eliminadas por distancia < 0.1 km: {eliminados_dist:,}")

df["fare_per_km"] = df["fare_amount"] / df["distance_km"]
mask_fpkm = df["fare_per_km"] <= 30
eliminados_fpkm = (~mask_fpkm).sum()
df = df[mask_fpkm].copy()
print(f"Filas eliminadas por fare_per_km > 30: {eliminados_fpkm:,}")

print("\nVariables temporales y distancia creadas exitosamente.")

### 4) Seleccion de variables para dashboard

In [ ]:
print_section("SELECCION DE VARIABLES PARA EL DASHBOARD")

DIMENSIONES = {
    "year": "Ano del viaje (temporal)",
    "month": "Mes del viaje (1-12)",
    "month_name": "Nombre del mes",
    "day_of_week": "Dia de la semana (0=Lunes ... 6=Domingo)",
    "day_name": "Nombre del dia",
    "hour": "Hora del dia (0-23)",
    "time_of_day": "Franja horaria",
}

METRICAS = {
    "fare_amount": "Tarifa cobrada en USD",
    "distance_km": "Distancia del viaje en km",
    "passenger_count": "Numero de pasajeros por viaje",
    "fare_per_km": "Tarifa por kilometro",
}

COORDENADAS = {
    "pickup_latitude": "Latitud de recogida",
    "pickup_longitude": "Longitud de recogida",
    "dropoff_latitude": "Latitud de destino",
    "dropoff_longitude": "Longitud de destino",
}

print("DIMENSIONES:")
for col, desc in DIMENSIONES.items():
    print(f" - {col:<18} : {desc}")

print("\nMETRICAS:")
for col, desc in METRICAS.items():
    print(f" - {col:<18} : {desc}")

print("\nCOORDENADAS:")
for col, desc in COORDENADAS.items():
    print(f" - {col:<18} : {desc}")

### 5) Dataframe final y export

In [ ]:
print_section("DATAFRAME FINAL PARA ANALISIS")

columnas_finales = list(DIMENSIONES.keys()) + list(METRICAS.keys()) + list(COORDENADAS.keys())
df_final = df[columnas_finales].reset_index(drop=True)

print("Head (5 filas):")
print(df_final.head().to_string(index=False))

print("\nShape del dataset limpio:")
print(f"{df_final.shape[0]:,} filas x {df_final.shape[1]} columnas")

print("\nEstadisticas descriptivas (metricas):")
print(df_final[list(METRICAS.keys())].describe().round(2).to_string())

df_final.to_csv("uber_clean.csv", index=False)
print("\nDataset limpio guardado como 'uber_clean.csv'")